# **REVERSE IMAGE SEARCH ENGINE - FEATURE EXTRACTION**

## IMPORTS

In [1]:
import numpy as np
from numpy.linalg import norm
import os
import random
import math

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model

## UTILITIES

In [4]:
def extract_features(img_path, feature_model):
    img = load_img(img_path, target_size=(img_width, img_height))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    features = feature_model.predict(img_array)
    flattened = features.flatten()
    normalized = flattened / norm(flattened)
    return normalized

In [5]:
extensions = ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']


def get_file_paths(root_dir):
    file_list = []
    for root, directories, filenames in os.walk(root_dir):
        for filename in filenames:
            if any(ext in filename for ext in extensions):
                filepath = os.path.join(root, filename)
                if os.path.exists(filepath):
                  file_list.append(filepath)
                else:
                  print(filepath)
    return file_list

In [6]:
def define_model():
    base_model = ResNet50(include_top=False, input_shape=(img_width, img_height, 3), pooling='avg')
    for layer in base_model.layers:
        layer.trainable = False

    input_tensor = Input(shape=(img_width, img_height, 3))
    x = base_model(input_tensor)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    output_tensor = Dense(num_classes, activation='softmax')(x)

    full_model = Model(inputs=input_tensor, outputs=output_tensor)
    return full_model, base_model

## FEATURE EXTRACTION

In [7]:
project_root = os.path.dirname(os.getcwd())
base_dir = os.path.join(project_root, "data", "caltech101")
root_dir = os.path.join(base_dir, "caltech-101", "101_ObjectCategories")

filenames = sorted(get_file_paths(root_dir))
print(f'There are {len(filenames)} files in the dataset.')

There are 8677 files in the dataset.


In [8]:
train_samples = 8677
num_classes = 101
img_width, img_height = 224, 224
batch_size = 128

In [9]:
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                   rotation_range=25,
                                   width_shift_range=0.15,
                                   height_shift_range=0.15,
                                   zoom_range=0.3)

In [10]:
train_generator = train_datagen.flow_from_directory(root_dir,
                                                    target_size=(img_width, img_height),
                                                    shuffle=True,
                                                    seed=10000,
                                                    class_mode='categorical')

Found 8677 images belonging to 101 classes.


In [11]:
num_images = len(train_generator.filenames)
steps_per_epochs = int(math.ceil(num_images / batch_size))
print(f'Number of images: {num_images}')
print(f'Number of steps per epochs: {steps_per_epochs}')

Number of images: 8677
Number of steps per epochs: 68


In [12]:
filenames = [root_dir + '/' + s for s in train_generator.filenames]

In [13]:
model_finetuned, feature_extractor = define_model()
model_finetuned.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(0.001),
    metrics=['acc']
)

model_finetuned.fit(
    train_generator,
    steps_per_epoch=steps_per_epochs,
    epochs=10
)

d:\Data Science\My Projects\Reverse Image Search Engine\venv\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 108s 2s/step - acc: 0.1607 - loss: 4.1010
Epoch 2/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 140s 2s/step - acc: 0.3901 - loss: 2.7515
Epoch 3/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 144s 2s/step - acc: 0.5066 - loss: 2.1383
Epoch 4/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 136s 2s/step - acc: 0.5725 - loss: 1.7655
Epoch 5/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 136s 2s/step - acc: 0.6201 - loss: 1.4730
Epoch 6/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 146s 2s/step - acc: 0.6609 - loss: 1.3583
Epoch 7/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 148s 2s/step - acc: 0.6681 - loss: 1.2668
Epoch 8/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 138s 2s/step - acc: 0.7052 - loss: 1.1105
Epoch 9/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - acc: 0.6982 - loss: 1.1054
Epoch 10/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 132s 2s/step - acc: 0.7417 - loss: 0.9695


In [ ]:
all_features = np.array([extract_features(img_path, feature_extractor) for img_path in filenames])

In [15]:
features_dir = os.path.join(project_root, "features_data")
os.makedirs(features_dir, exist_ok=True)

In [16]:
np.save(os.path.join(features_dir, "features.npy"), all_features)
np.save(os.path.join(features_dir, "filenames.npy"), filenames)

class_ids = train_generator.classes
np.save(os.path.join(features_dir, "class_ids.npy"), class_ids)

print("Saved features, filenames, and class IDs to features_data directory.")

Saved features, filenames, and class IDs to features_data directory.
